In [1]:
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from astropy.visualization import simple_norm
import astropy.units as u
from reproject import reproject_interp
import warnings
warnings.filterwarnings('ignore')

MIRI_VEGA_ZP_JY = {
    'f770w':  64.13,
    'f1130w': 29.40,
    'f1500w': 17.60,
    'f2550w':  6.72,
}

WAVELENGTHS_UM = {
    'f115w': 1.154,
    'f182m': 1.845,
    'f187n': 1.874,
    'f200w': 1.989,
    'f212n': 2.121,
    'f356w': 3.563,
    'f405n': 4.052,
    'f410m': 4.082,
    'f444w': 4.421,
    'f466n': 4.654,
    'f770w': 7.639,
    'f1130w': 11.309,
    'f1500w': 15.065,
    'f2550w': 25.363,
}

NIRCAM_FILTERS = ['f115w', 'f182m', 'f187n', 'f200w', 'f212n',
                  'f356w', 'f405n', 'f410m', 'f444w', 'f466n']

BRICK_BASE   = Path('/blue/adamginsburg/adamginsburg/jwst/brick')
SICKLE_BASE  = Path('/orange/adamginsburg/jwst/sickle')
BRICK_ORANGE = Path('/orange/adamginsburg/jwst/brick')
IMGS_DIR     = BRICK_BASE / 'images'
INPAINT_DIR  = IMGS_DIR / 'inpainted'

MIRI_CAT_FILE   = SICKLE_BASE / 'catalogs' / 'o003_miri_cmd_matched.fits'
NIRCAM_CAT_FILE = BRICK_ORANGE / 'catalogs/basic_merged_indivexp_photometry_tables_merged.fits'

IMAGE_FILES = {
    'f115w': INPAINT_DIR / 'jw01182-o004_t001_nircam_clear-f115w-merged_i2d_inpainted.fits',
    'f200w': IMGS_DIR   / 'jw01182-o004_t001_nircam_clear-f200w-merged_i2d.fits',
    'f356w': INPAINT_DIR / 'jw01182-o004_t001_nircam_clear-f356w-merged_i2d_inpainted.fits',
    'f444w': INPAINT_DIR / 'jw01182-o004_t001_nircam_clear-f444w-merged_i2d_inpainted.fits',
    'f182m': INPAINT_DIR / 'jw02221-o001_t001_nircam_clear-f182m-merged-reproject_i2d_inpainted.fits',
    'f187n': INPAINT_DIR / 'jw02221-o001_t001_nircam_clear-f187n-merged-reproject_i2d_inpainted.fits',
    'f212n': INPAINT_DIR / 'jw02221-o001_t001_nircam_clear-f212n-merged-reproject_i2d_inpainted.fits',
    'f405n': INPAINT_DIR / 'jw02221-o001_t001_nircam_clear-f405n-merged-reproject_i2d_inpainted.fits',
    'f410m': INPAINT_DIR / 'jw02221-o001_t001_nircam_clear-f410m-merged-reproject_i2d_inpainted.fits',
    'f466n': INPAINT_DIR / 'jw02221-o001_t001_nircam_clear-f466n-merged-reproject_i2d_inpainted.fits',
    'f770w':  SICKLE_BASE / 'F770W/pipeline/jw03958-o003_t001_miri_f770w_i2d.fits',
    'f1130w': SICKLE_BASE / 'F1130W/pipeline/jw03958-o003_t001_miri_f1130w_i2d.fits',
    'f1500w': SICKLE_BASE / 'F1500W/pipeline/jw03958-o003_t001_miri_f1500w_i2d.fits',
    'f2550w': BRICK_ORANGE / 'F2550W/pipeline/jw02221-o002_t001_miri_f2550w_i2d.fits',
}

OUT_DIR = BRICK_BASE / 'sed_figures_f1500w_all'
OUT_DIR.mkdir(exist_ok=True)

for k, v in IMAGE_FILES.items():
    print(f'{k}: {"OK" if v.exists() else "MISSING"}  {v}')


f115w: OK  /blue/adamginsburg/adamginsburg/jwst/brick/images/inpainted/jw01182-o004_t001_nircam_clear-f115w-merged_i2d_inpainted.fits
f200w: OK  /blue/adamginsburg/adamginsburg/jwst/brick/images/jw01182-o004_t001_nircam_clear-f200w-merged_i2d.fits
f356w: OK  /blue/adamginsburg/adamginsburg/jwst/brick/images/inpainted/jw01182-o004_t001_nircam_clear-f356w-merged_i2d_inpainted.fits
f444w: OK  /blue/adamginsburg/adamginsburg/jwst/brick/images/inpainted/jw01182-o004_t001_nircam_clear-f444w-merged_i2d_inpainted.fits
f182m: OK  /blue/adamginsburg/adamginsburg/jwst/brick/images/inpainted/jw02221-o001_t001_nircam_clear-f182m-merged-reproject_i2d_inpainted.fits
f187n: OK  /blue/adamginsburg/adamginsburg/jwst/brick/images/inpainted/jw02221-o001_t001_nircam_clear-f187n-merged-reproject_i2d_inpainted.fits
f212n: OK  /blue/adamginsburg/adamginsburg/jwst/brick/images/inpainted/jw02221-o001_t001_nircam_clear-f212n-merged-reproject_i2d_inpainted.fits
f405n: OK  /blue/adamginsburg/adamginsburg/jwst/bric

In [2]:
# Load MIRI o003 catalog (brick footprint only)
miri_cat = Table.read(MIRI_CAT_FILE)
print(f'MIRI o003 sources: {len(miri_cat)}')

for col in list(miri_cat.colnames):
    if col != col.lower():
        miri_cat.rename_column(col, col.lower())

def _to_float(val):
    if val is None: return np.nan
    if hasattr(val, 'mask') and val.mask: return np.nan
    try:
        v = float(val)
        return v if np.isfinite(v) else np.nan
    except (TypeError, ValueError): return np.nan

mag_f15 = np.array([_to_float(v) for v in miri_cat['mag_f1500w']])
print(f'F1500W valid: {np.sum(np.isfinite(mag_f15))} / {len(mag_f15)}')

# Load large unfiltered NIRCam catalog (2.9 GB).
# Open raw FITS with memmap to read RA/Dec quickly, then spatially filter.
print(f'\nOpening {NIRCAM_CAT_FILE.name} with memmap...')
_RA_MIN, _RA_MAX   = 266.35, 266.75
_DEC_MIN, _DEC_MAX = -28.85, -28.55
with fits.open(NIRCAM_CAT_FILE, memmap=True) as _hdul:
    _raw_ra  = np.array(_hdul[1].data['skycoord_ref.ra'])
    _raw_dec = np.array(_hdul[1].data['skycoord_ref.dec'])
    _bmask   = ((_raw_ra  >= _RA_MIN)  & (_raw_ra  <= _RA_MAX) &
                (_raw_dec >= _DEC_MIN) & (_raw_dec <= _DEC_MAX))
    print(f'Brick pre-filter: {_bmask.sum():,} / {len(_raw_ra):,} rows')
    nircam_cat = Table(_hdul[1].data[_bmask])   # reads matching rows while hdul open

print(f'NIRCam loaded: {len(nircam_cat):,} rows, {len(nircam_cat.colnames)} cols')
print(f'First 5 cols: {nircam_cat.colnames[:5]}')

# Detect whether RA/Dec are plain float columns (raw FITS path) or SkyCoord mixin
if 'skycoord_ref.ra' in nircam_cat.colnames:
    _NC_RA_COL, _NC_DEC_COL = 'skycoord_ref.ra', 'skycoord_ref.dec'
    nircam_coords = SkyCoord(
        nircam_cat[_NC_RA_COL].astype(float) * u.deg,
        nircam_cat[_NC_DEC_COL].astype(float) * u.deg,
    )
    print('RA/Dec: plain float columns')
elif 'skycoord_ref' in nircam_cat.colnames:
    _NC_RA_COL, _NC_DEC_COL = None, None   # mixin — handled separately
    _sc = nircam_cat['skycoord_ref']
    nircam_coords = SkyCoord(_sc.ra, _sc.dec)
    print('RA/Dec: SkyCoord mixin column')
else:
    raise RuntimeError(f'Cannot find RA/Dec in NIRCam catalog. Cols: {nircam_cat.colnames[:10]}')

print(f'NIRCam RA:  {nircam_coords.ra.deg.min():.4f} – {nircam_coords.ra.deg.max():.4f}')
print(f'NIRCam Dec: {nircam_coords.dec.deg.min():.4f} – {nircam_coords.dec.deg.max():.4f}')


MIRI o003 sources: 1776


F1500W valid: 241 / 1776

Opening basic_merged_indivexp_photometry_tables_merged.fits with memmap...


Brick pre-filter: 1,958,951 / 1,958,958 rows


NIRCam loaded: 1,958,951 rows, 295 cols
First 5 cols: ['skycoord_ref.ra', 'skycoord_ref.dec', 'skycoord_ref_filtername', 'sep_f410m', 'id_f410m']


RA/Dec: plain float columns


NIRCam RA:  266.4733 – 266.6512
NIRCam Dec: -28.8336 – -28.5671


In [ ]:
# Load consolidated satstar catalogs + PIXAR_SR per filter
SATSTAR_CAT_DIR = BRICK_BASE / 'catalogs'

SATSTAR_CATS   = {}   # filt -> Table
SATSTAR_COORDS = {}   # filt -> SkyCoord
SATSTAR_PIX_SR = {}   # filt -> pixel area in sr
_SATSTAR_RADIUS_ARCSEC = 0.5

for _filt in NIRCAM_FILTERS + ['f2550w']:
    _cat_path = SATSTAR_CAT_DIR / f'{_filt}_consolidated_satstar_catalog.fits'
    if not _cat_path.exists():
        print(f'{_filt}: no satstar catalog')
        continue
    _cat = Table.read(str(_cat_path))
    SATSTAR_CATS[_filt] = _cat
    _sc = _cat['skycoord_fit']
    SATSTAR_COORDS[_filt] = SkyCoord(_sc.ra.deg * u.deg, _sc.dec.deg * u.deg)
    _img_path = IMAGE_FILES.get(_filt)
    if _img_path is None or not _img_path.exists():
        print(f'{_filt}: image missing, cannot get PIXAR_SR')
        continue
    _pixar = None
    with fits.open(str(_img_path)) as _hdul:
        for _ext in [('SCI', 1), 1, 0]:
            try:
                _pixar = _hdul[_ext].header.get('PIXAR_SR')
                if _pixar is not None:
                    break
            except Exception:
                pass
        if _pixar is None:
            for _ext in [1, 0]:
                try:
                    _ww = WCS(_hdul[_ext].header)
                    _pixar = float(_ww.proj_plane_pixel_area().to(u.sr).value)
                    break
                except Exception:
                    pass
    if _pixar is not None:
        SATSTAR_PIX_SR[_filt] = float(_pixar)
    print(f'{_filt}: {len(_cat)} satstar sources, PIXAR_SR={SATSTAR_PIX_SR.get(_filt, float("nan")):.4e}')


In [3]:
# All sources with valid F1500W (sorted bright-to-faint)
valid_mask  = np.isfinite(mag_f15)
miri_valid  = miri_cat[valid_mask]
mags_valid  = mag_f15[valid_mask]

sort_idx     = np.argsort(mags_valid)
sources      = miri_valid[sort_idx]
source_mags  = mags_valid[sort_idx]

print(f'F1500W sources: {len(sources)}')
print(f'Mag range: {source_mags[0]:.3f} – {source_mags[-1]:.3f} Vega')
source_coords = SkyCoord(sources['ra'] * u.deg, sources['dec'] * u.deg)


F1500W sources: 241
Mag range: 9.041 – 19.968 Vega


In [4]:
XMATCH_RADIUS = 0.5 * u.arcsec
idx_nc, sep_nc, _ = match_coordinates_sky(source_coords, nircam_coords)
has_nircam = sep_nc < XMATCH_RADIUS
print(f'NIRCam matches: {has_nircam.sum()} / {len(sources)}  (within {XMATCH_RADIUS})')


NIRCam matches: 241 / 241  (within 0.5 arcsec)


In [5]:
import pandas as pd
import numpy as np

# SESHAT results pre-computed by /tmp/run_seshat_brick.py (run standalone to avoid
# network-related hangs inside nbconvert kernel: requests.head() DNS timeout can
# exceed nbconvert's per-cell budget).
_SESHAT_RESULTS_FILE = BRICK_BASE / 'seshat_results_f1500w_all.csv'

if _SESHAT_RESULTS_FILE.exists():
    seshat_result = pd.read_csv(_SESHAT_RESULTS_FILE)
    seshat_classes = [c.replace('Prob ', '') for c in seshat_result.columns
                      if c.startswith('Prob ')]
    print(f'Loaded SESHAT results: {len(seshat_result)} rows')
    print(f'Classes: {seshat_classes}')
    print(seshat_result['Predicted_Class'].value_counts().to_string())
else:
    raise FileNotFoundError(
        f'{_SESHAT_RESULTS_FILE} not found. '
        'Run /tmp/run_seshat_brick.py first:\n'
        '  /orange/adamginsburg/miniconda3/envs/python313/bin/python3 /tmp/run_seshat_brick.py'
    )


Loaded SESHAT results: 241 rows
Classes: ['YSO', 'FS', 'WD', 'BD', 'Gal']
Predicted_Class
FS     122
YSO    112
Gal      5
BD       2


In [6]:
# Load images (memory-mapped)
image_data = {}
for filt, path in IMAGE_FILES.items():
    if not path.exists():
        print(f'SKIP (not found): {filt}')
        continue
    try:
        hdul = fits.open(path, memmap=True)
        data = hdul['SCI'].data
        wcs  = WCS(hdul['SCI'].header, naxis=2)
        image_data[filt] = (data, wcs)
        print(f'Loaded {filt}: {data.shape}')
    except Exception as e:
        print(f'ERROR loading {filt}: {e}')

Loaded f115w: (8213, 11793)
Loaded f200w: (8219, 11796)


Loaded f356w: (4003, 5828)
Loaded f444w: (4003, 5827)


Loaded f182m: (11769, 4854)


Loaded f187n: (11602, 4835)


Loaded f212n: (11602, 4835)


Loaded f405n: (5726, 2353)


Loaded f410m: (5727, 2353)


Loaded f466n: (5726, 2352)
Loaded f770w: (610, 683)


Loaded f1130w: (610, 683)
Loaded f1500w: (610, 683)


Loaded f2550w: (1058, 2858)


In [ ]:
def vega_mag_to_jy(mag, filt):
    zp = MIRI_VEGA_ZP_JY.get(filt.lower())
    m = _to_float(mag)
    if zp is None or not np.isfinite(m): return np.nan
    return zp * 10.0 ** (-m / 2.5)

SATSTAR_FALLBACK = {
    'f182m': ('flux_jy_182m187', WAVELENGTHS_UM['f182m']),
    'f187n': ('flux_jy_187m182', WAVELENGTHS_UM['f187n']),
    'f405n': ('flux_jy_405m410', WAVELENGTHS_UM['f405n']),
    'f410m': ('flux_jy_410m405', WAVELENGTHS_UM['f410m']),
}

PHOT_APER_ARCSEC = {
    'f115w': 0.15, 'f182m': 0.15, 'f187n': 0.15, 'f200w': 0.15, 'f212n': 0.15,
    'f356w': 0.25, 'f405n': 0.25, 'f410m': 0.25, 'f444w': 0.25, 'f466n': 0.25,
    'f770w': 0.50, 'f1130w': 0.75, 'f1500w': 1.00, 'f2550w': 1.50,
}

def measure_flux_image(coord, filt):
    arr, ps_arcsec = get_cutout(coord, filt)
    if arr is None or ps_arcsec is None:
        return np.nan
    ap_arcsec   = PHOT_APER_ARCSEC.get(filt, 0.5)
    ap_pix      = ap_arcsec / ps_arcsec
    ny, nx      = arr.shape
    cy, cx      = (ny - 1) / 2.0, (nx - 1) / 2.0
    yy, xx      = np.mgrid[:ny, :nx].astype(float)
    r           = np.sqrt((xx - cx)**2 + (yy - cy)**2)
    ap_mask     = (r <= ap_pix) & np.isfinite(arr)
    bg_mask     = (r > ap_pix * 2) & (r <= ap_pix * 3) & np.isfinite(arr)
    ap_vals     = arr[ap_mask]
    bg_vals     = arr[bg_mask]
    if len(ap_vals) < 3:
        return np.nan
    bg_per_pix  = float(np.nanmedian(bg_vals)) if len(bg_vals) >= 3 else 0.0
    net_sum     = float(np.sum(ap_vals - bg_per_pix))
    ps_rad      = ps_arcsec * np.pi / (180.0 * 3600.0)
    pix_area_sr = ps_rad ** 2
    flux_jy     = net_sum * pix_area_sr * 1e6
    return float(flux_jy) if np.isfinite(flux_jy) and flux_jy > 0 else np.nan


def get_satstar_flux(filt, coord):
    if filt not in SATSTAR_COORDS or filt not in SATSTAR_PIX_SR:
        return np.nan, np.nan
    idx, sep, _ = coord.match_to_catalog_sky(SATSTAR_COORDS[filt])
    if float(sep.arcsec) > _SATSTAR_RADIUS_ARCSEC:
        return np.nan, np.nan
    cat = SATSTAR_CATS[filt]
    row = cat[int(idx)]
    pixar_sr = SATSTAR_PIX_SR[filt]
    flux_jy = float(row['flux_fit']) * pixar_sr * 1e6
    if not (np.isfinite(flux_jy) and flux_jy > 0):
        return np.nan, np.nan
    _err_col = ('flux_err' if 'flux_err' in cat.colnames
                else 'flux_unc' if 'flux_unc' in cat.colnames else None)
    eflux_jy = float(row[_err_col]) * pixar_sr * 1e6 if _err_col is not None else np.nan
    return float(flux_jy), float(eflux_jy)


def get_source_fluxes(i):
    row           = sources[i]
    miri_coord    = source_coords[i]
    nc_coord      = get_nircam_coord(i)
    fluxes        = {}
    nc_phot_coord = nc_coord if nc_coord is not None else miri_coord

    # MIRI f770w, f1130w, f1500w: catalog magnitudes
    for filt in ['f770w', 'f1130w', 'f1500w']:
        col = f'mag_{filt}'
        if col not in miri_cat.colnames: continue
        f = vega_mag_to_jy(_to_float(row[col]), filt)
        if np.isfinite(f) and f > 0:
            fluxes[filt] = (WAVELENGTHS_UM[filt], f, np.nan, False)

    # f2550w: satstar -> miri_cat -> miri_cat forced -> image
    sat_f, sat_ef = get_satstar_flux('f2550w', miri_coord)
    if np.isfinite(sat_f):
        fluxes['f2550w'] = (WAVELENGTHS_UM['f2550w'], sat_f, sat_ef, False)
    else:
        _col = 'mag_f2550w'
        if _col in miri_cat.colnames:
            f = vega_mag_to_jy(_to_float(row[_col]), 'f2550w')
            if np.isfinite(f) and f > 0:
                fluxes['f2550w'] = (WAVELENGTHS_UM['f2550w'], f, np.nan, False)
        if 'f2550w' not in fluxes and 'mag_f2550w_forced' in miri_cat.colnames:
            f = vega_mag_to_jy(_to_float(row['mag_f2550w_forced']), 'f2550w')
            if np.isfinite(f) and f > 0:
                fluxes['f2550w'] = (WAVELENGTHS_UM['f2550w'], f, np.nan, False)
        if 'f2550w' not in fluxes:
            f = measure_flux_image(miri_coord, 'f2550w')
            if np.isfinite(f):
                fluxes['f2550w'] = (WAVELENGTHS_UM['f2550w'], f, np.nan, True)

    # NIRCam filters: satstar (primary) -> nircam_cat -> image
    nc_row = nircam_cat[int(idx_nc[i])] if has_nircam[i] else None
    for filt in NIRCAM_FILTERS:
        # 1. Satstar catalog (latest recovered photometry)
        sat_f, sat_ef = get_satstar_flux(filt, miri_coord)
        if np.isfinite(sat_f):
            fluxes[filt] = (WAVELENGTHS_UM[filt], sat_f, sat_ef, False)
            continue
        # 2. nircam_cat merged catalog
        if nc_row is not None:
            fcol = f'flux_jy_{filt}'
            ecol = f'eflux_jy_{filt}'
            f  = _to_float(nc_row[fcol]) if fcol in nircam_cat.colnames else np.nan
            ef = _to_float(nc_row[ecol]) if ecol in nircam_cat.colnames else np.nan
            if np.isfinite(f) and f > 0:
                fluxes[filt] = (WAVELENGTHS_UM[filt], f, ef, False)
                continue
            fb = SATSTAR_FALLBACK.get(filt)
            if fb is not None and fb[0] in nircam_cat.colnames:
                f2 = _to_float(nc_row[fb[0]])
                if np.isfinite(f2) and f2 > 0:
                    fluxes[filt] = (fb[1], f2, np.nan, False)
                    continue
        # 3. Image aperture fallback
        f = measure_flux_image(nc_phot_coord, filt)
        if np.isfinite(f):
            fluxes[filt] = (WAVELENGTHS_UM[filt], f, np.nan, True)

    return dict(sorted(fluxes.items(), key=lambda kv: kv[1][0]))


def get_nircam_coord(i):
    if not has_nircam[i]:
        return None
    nc_row = nircam_cat[int(idx_nc[i])]
    if _NC_RA_COL is not None:
        try:
            return SkyCoord(float(nc_row[_NC_RA_COL]) * u.deg,
                            float(nc_row[_NC_DEC_COL]) * u.deg)
        except (KeyError, TypeError, ValueError):
            return None
    try:
        sc = nc_row['skycoord_ref']
        return SkyCoord(float(sc.ra.deg) * u.deg, float(sc.dec.deg) * u.deg)
    except (KeyError, AttributeError, TypeError):
        return None


In [8]:
CUTOUT_SIZE     = 5.0 * u.arcsec   # final displayed size (all filters)
CUTOUT_PAD      = 8.0 * u.arcsec   # padded initial cutout to cover rotated footprint

# 14 filters in wavelength order → 3×5 grid (slot [2,4] empty)
CUTOUT_FILTERS = [
    'f115w', 'f182m', 'f187n', 'f200w', 'f212n',   # row 0
    'f356w', 'f405n', 'f410m', 'f444w', 'f466n',   # row 1
    'f770w', 'f1130w', 'f1500w', 'f2550w',          # row 2 (slot 4 empty)
]
N_COLS = 5

def _make_northup_wcs(coord, pixscale_deg, npix):
    """Simple north-up, east-left TAN WCS centred on coord."""
    w = WCS(naxis=2)
    w.wcs.crpix = [npix / 2 + 0.5, npix / 2 + 0.5]
    w.wcs.cdelt = [-pixscale_deg, pixscale_deg]   # RA decreases left→right
    w.wcs.crval = [coord.ra.deg, coord.dec.deg]
    w.wcs.ctype = ['RA---TAN', 'DEC--TAN']
    return w

def get_cutout(coord, filt):
    """Return (arr, pixscale_arcsec) reprojected to north-up, or (None, None)."""
    if filt not in image_data:
        return None, None
    data, wcs = image_data[filt]
    try:
        # Native pixel scale
        pm = wcs.pixel_scale_matrix
        ps_deg = np.sqrt(abs(np.linalg.det(pm)))
        ps_arcsec = ps_deg * 3600.0

        # Padded cutout in native pixel frame (covers any rotation)
        cut = Cutout2D(data, coord, CUTOUT_PAD, wcs=wcs,
                       mode='partial', fill_value=np.nan)
        if np.all(~np.isfinite(cut.data)):
            return None, None

        # Reproject to north-up TAN WCS of the same pixel scale
        size_deg = CUTOUT_SIZE.to(u.deg).value
        npix = max(5, int(round(size_deg / ps_deg)))
        target_wcs = _make_northup_wcs(coord, ps_deg, npix)

        arr, footprint = reproject_interp(
            (cut.data, cut.wcs), target_wcs,
            shape_out=(npix, npix), order='bilinear',
        )
        arr = np.array(arr, dtype=float)
        arr[footprint < 0.5] = np.nan   # mask outside-footprint pixels

        if np.all(~np.isfinite(arr)) or np.nansum(np.abs(arr)) == 0:
            return None, None
        return arr, ps_arcsec
    except Exception:
        return None, None

In [9]:
from matplotlib import gridspec
from PIL import Image, PngImagePlugin

MIRI_FILTS = set(MIRI_VEGA_ZP_JY.keys())

def plot_source_sed(i_src, save=True):
    miri_coord = source_coords[i_src]
    nc_coord   = get_nircam_coord(i_src)
    rank       = i_src + 1
    mag15      = source_mags[i_src]
    fluxes     = get_source_fluxes(i_src)

    n_cat = sum(1 for v in fluxes.values() if not v[3])
    n_img = sum(1 for v in fluxes.values() if     v[3])

    # SESHAT classification for this source
    sr        = seshat_result.iloc[i_src]
    cls       = str(sr['Predicted_Class'])
    cls_prob  = float(sr[f'Prob {cls}'])

    fig = plt.figure(figsize=(14, 11))
    fig.suptitle(
        f'Rank {rank}  |  RA={miri_coord.ra.deg:.5f}  Dec={miri_coord.dec.deg:.5f}  '
        f'|  F1500W={mag15:.3f} Vega  '
        f'|  SESHAT: {cls} ({cls_prob:.0%})  '
        f'|  {n_cat} cat + {n_img} img-phot',
        fontsize=10, y=1.00,
    )

    gs = gridspec.GridSpec(
        4, N_COLS,
        height_ratios=[1, 1, 1, 1.4],
        hspace=0.35, wspace=0.08,
        top=0.96, bottom=0.06, left=0.04, right=0.98,
    )

    for idx, filt in enumerate(CUTOUT_FILTERS):
        row_g = idx // N_COLS
        col_g = idx %  N_COLS
        ax    = fig.add_subplot(gs[row_g, col_g])

        coord = miri_coord if filt in MIRI_FILTS else (nc_coord if nc_coord is not None else miri_coord)
        arr, pix_scale = get_cutout(coord, filt)

        if arr is not None:
            finite = arr[np.isfinite(arr)]
            if finite.size > 0 and finite.max() > finite.min():
                norm = simple_norm(finite, stretch='log', percent=99.5)
            else:
                norm = None
            ax.imshow(arr, norm=norm, origin='lower', cmap='inferno')
            cx_s, cy_s = arr.shape[1] / 2, arr.shape[0] / 2
            ap_pix = PHOT_APER_ARCSEC.get(filt, 0.5) / pix_scale
            ax.add_patch(plt.Circle((cx_s, cy_s), ap_pix,
                                    color='cyan', fill=False, lw=0.8))
            ax.axhline(cy_s, color='cyan', lw=0.4, alpha=0.5)
            ax.axvline(cx_s, color='cyan', lw=0.4, alpha=0.5)
        else:
            ax.set_facecolor('#111')
            ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                    transform=ax.transAxes, fontsize=7, color='gray')

        flux_entry = fluxes.get(filt)
        clr = ('orange' if flux_entry[3] else 'lime') if flux_entry is not None else 'white'
        ax.set_title(filt.upper(), fontsize=8, pad=2, color=clr)
        ax.set_xticks([])
        ax.set_yticks([])

    ax_sed = fig.add_subplot(gs[3, :])

    if fluxes:
        cat_items = [(k, v) for k, v in fluxes.items() if not v[3]]
        img_items = [(k, v) for k, v in fluxes.items() if     v[3]]

        if cat_items:
            wc  = np.array([v[0] for _, v in cat_items])
            fc  = np.array([v[1] for _, v in cat_items])
            efc = np.array([v[2] for _, v in cat_items])
            ax_sed.plot(wc, fc, '-', color='steelblue', lw=0.8, alpha=0.4, zorder=2)
            ax_sed.errorbar(wc, fc,
                            yerr=np.where(np.isfinite(efc), efc, 0),
                            fmt='o', color='steelblue', ms=5, capsize=3, lw=1, zorder=3,
                            label='catalog')
            for w, f, lbl in zip(wc, fc, [k for k, _ in cat_items]):
                ax_sed.annotate(lbl.upper(), (w, f),
                                textcoords='offset points', xytext=(0, 6),
                                ha='center', fontsize=7)

        if img_items:
            wi = np.array([v[0] for _, v in img_items])
            fi = np.array([v[1] for _, v in img_items])
            ax_sed.plot(wi, fi, '--', color='orange', lw=0.6, alpha=0.4, zorder=2)
            ax_sed.errorbar(wi, fi, fmt='^', color='orange', ms=5, lw=1, zorder=3,
                            label='image aperture (no ap-corr)')
            for w, f, lbl in zip(wi, fi, [k for k, _ in img_items]):
                ax_sed.annotate(lbl.upper(), (w, f),
                                textcoords='offset points', xytext=(0, 6),
                                ha='center', fontsize=7, color='orange')

        ax_sed.legend(fontsize=8, loc='upper left')

    ax_sed.set_xscale('log')
    ax_sed.set_yscale('log')
    ax_sed.set_xlabel(r'Wavelength ($\mu$m)', fontsize=9)
    ax_sed.set_ylabel('Flux (Jy)', fontsize=9)
    ax_sed.grid(True, alpha=0.3)

    if save:
        outname = OUT_DIR / f'sed_rank{rank:03d}_F1500W{mag15:.2f}.png'
        fig.savefig(outname, dpi=150, bbox_inches='tight')
        plt.close(fig)

        # Embed SESHAT classification in PNG tEXt metadata
        img  = Image.open(outname)
        meta = PngImagePlugin.PngInfo()
        meta.add_text('SESHAT_CLASS', cls)
        meta.add_text('SESHAT_PROB',  f'{cls_prob:.6f}')
        for c in seshat_classes:
            meta.add_text(f'SESHAT_PROB_{c}', f'{float(sr[f"Prob {c}"]):.6f}')
        img.save(outname, pnginfo=meta)

        return outname
    return fig


In [10]:
sources.write(OUT_DIR / 'all_f1500w_sources_summary.fits', overwrite=True)

n_src = len(sources)
for i in range(n_src):
    outname = plot_source_sed(i)
    if (i + 1) % 25 == 0 or (i + 1) == n_src:
        print(f'{i+1}/{n_src} done: {outname.name}')

print('Done. Figures in', OUT_DIR)


25/241 done: sed_rank025_F1500W11.60.png


50/241 done: sed_rank050_F1500W12.40.png


75/241 done: sed_rank075_F1500W12.84.png


100/241 done: sed_rank100_F1500W13.19.png


125/241 done: sed_rank125_F1500W13.52.png


150/241 done: sed_rank150_F1500W13.80.png


175/241 done: sed_rank175_F1500W14.06.png


200/241 done: sed_rank200_F1500W14.42.png


225/241 done: sed_rank225_F1500W15.29.png


241/241 done: sed_rank241_F1500W19.97.png
Done. Figures in /blue/adamginsburg/adamginsburg/jwst/brick/sed_figures_f1500w_all
